## The Problem: Large Movie Dataset Review
### Classify movie reviews from IMDB into positive or negative sentiment.
### Download the dataset [here](https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz)

In [17]:
# imports

from gensim.models import KeyedVectors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import text_dataset_from_directory
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, Dense, Input, GlobalAveragePooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

import todo as utils
from pprint import pprint

In [2]:
print(tf.__version__)

2.20.0


## Exploring the data

In [3]:
# Importing & preprocessing the dataset

train_ds = text_dataset_from_directory("../../imdb/train")
test_ds = text_dataset_from_directory("../../imdb/test")

dfTrain = pd.DataFrame(
    train_ds.unbatch().as_numpy_iterator(), columns=["text", "label"]
)
dfTest = pd.DataFrame(test_ds.unbatch().as_numpy_iterator(), columns=["text", "label"])
_, xts = train_test_split(dfTest, stratify=dfTest["label"], test_size=0.25)

dfTrain["text"] = dfTrain["text"].map(lambda x: x.decode())
xts["text"] = xts["text"].map(lambda x: x.decode())

Found 75000 files belonging to 3 classes.
Found 25000 files belonging to 2 classes.


2025-11-16 14:24:50.060395: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-11-16 14:24:51.043331: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
pd.options.display.max_colwidth = 200
dfTrain.sample(n=5)

,text,label
24362,"My husband thought it would be fun to watch a bad Halloween-type movie with some unsuspecting friends. After all, we got the DVD on our honeymoon. To Walt Disney World, no less.<br /><br />But fun...",2
16504,"Historically wrong, but the plot is great. Even if the historical facts are wrong the movie is quite good. The movie tells the story of Snapphanarna in Skåne (the southest ""state"" of Sweden"") and ...",2
34205,"Okay, here's what I think about Jack Frost. I looked at the morphing box for the VHS tape and I thought to myself, this looks interesting. I rent it and I take it home. Boy was I right, it is inte...",1
65610,"On the plus side, ""Camp"" has a lot of energy and loads of affection for it's young characters. Some of the musical numbers are fun (""Turkey Lurkey Time"" from ""Promises, Promises""; the funny All Ab...",2
19383,"I saw this film with two senior citizens; I'd picked it out from the video store -- which I did frequently for these friends. The husband was in his 80s, his wife her late 60s; both love the theat...",2


In [19]:
print(dfTrain["text"][0].split("\n"))

["Please Don't hate me but i have to be honest, watching this movie, i had a lot of fun,,<br /><br />It's a Movie with a Stupid Cast and Stupid Songs!!!<br /><br />Unnecessary songs!!! Mehbooba... A Total Insult to the Original One Holi.... well.. it was OK! due to the Tradition Every Movie got to have one!! Chad Raha hai Nasha Whatever... Very UNNEEDED stupid Song jee Le... Sounded like a Playboy Song Stupid Song...<br /><br />Other than Songs. The Movie was OK This was Ram Gopal Verma's Own Adaptation... If you think like that you will like this movie<br /><br />Well this movie only Depends on the Viewer and on his judgement whether he/she thinks this movie is total Copy He/She would want to hit her Head on the Cinema Seat OR if he/She Thinks of the Directors Own Look he/she would be relaxed and take a look at this movie<br /><br />Anyways I looked at both ways i would Congratulate and Abuse Ram Gopal for this Disaster that he made...<br /><br />Well Some other Things that bothered t

In [21]:
len(dfTrain["text"])

75000

## Tokenize the text

In [15]:
dfTrain["label"].value_counts()

label
2    50000
0    12500
1    12500
Name: count, dtype: int64

In [6]:
tokenizer = Tokenizer()  # Deprecated
tokenizer.fit_on_texts(dfTrain["text"].tolist())
train_sequences = tokenizer.texts_to_sequences(dfTrain["text"].tolist())
test_sequences = tokenizer.texts_to_sequences(xts["text"].tolist())


word_index = tokenizer.word_index
print("Found %s unique tokens." % len(word_index))

Found 153845 unique tokens.


In [7]:
print(train_sequences[0])

[594, 89, 734, 70, 18, 10, 25, 5, 26, 1157, 147, 11, 17, 10, 67, 3, 169, 4, 248, 7, 7, 43, 3, 17, 15, 3, 375, 174, 2, 375, 768, 7, 7, 1793, 768, 38341, 3, 942, 2422, 5, 1, 207, 27, 28331, 69, 9, 13, 592, 697, 5, 1, 3036, 172, 17, 192, 5, 25, 27, 7375, 87409, 8459, 58547, 834, 52, 17136, 375, 652, 58548, 3275, 2920, 37, 3, 5347, 652, 375, 652, 7, 7, 84, 71, 768, 1, 17, 13, 592, 11, 13, 10052, 17137, 40751, 197, 1457, 45, 22, 102, 37, 12, 22, 82, 37, 11, 17, 7, 7, 69, 11, 17, 62, 6005, 20, 1, 534, 2, 20, 24, 9565, 770, 28, 57, 1200, 11, 17, 6, 942, 1015, 28, 57, 58, 180, 5, 612, 38, 414, 20, 1, 451, 2054, 39, 45, 28, 57, 1200, 4, 1, 981, 197, 166, 28, 57, 58, 26, 8240, 2, 189, 3, 166, 30, 11, 17, 7, 7, 3916, 10, 583, 30, 193, 744, 10, 58, 14952, 2, 2522, 10052, 17137, 16, 11, 1544, 12, 28, 91, 7, 7, 69, 46, 84, 177, 12, 2612, 12, 1, 174, 13, 983, 74, 7, 7, 6080, 7702, 14, 26580, 33006, 6080, 8, 24, 197, 17, 993, 152, 48, 13, 1, 165, 533, 11803, 16393, 14, 23810, 47364, 3577, 16200, 97, 2

In [8]:
print([tokenizer.index_word[k] for k in train_sequences[0]])

['please', "don't", 'hate', 'me', 'but', 'i', 'have', 'to', 'be', 'honest', 'watching', 'this', 'movie', 'i', 'had', 'a', 'lot', 'of', 'fun', 'br', 'br', "it's", 'a', 'movie', 'with', 'a', 'stupid', 'cast', 'and', 'stupid', 'songs', 'br', 'br', 'unnecessary', 'songs', 'mehbooba', 'a', 'total', 'insult', 'to', 'the', 'original', 'one', 'holi', 'well', 'it', 'was', 'ok', 'due', 'to', 'the', 'tradition', 'every', 'movie', 'got', 'to', 'have', 'one', 'chad', 'raha', 'hai', 'nasha', 'whatever', 'very', 'unneeded', 'stupid', 'song', 'jee', 'le', 'sounded', 'like', 'a', 'playboy', 'song', 'stupid', 'song', 'br', 'br', 'other', 'than', 'songs', 'the', 'movie', 'was', 'ok', 'this', 'was', 'ram', 'gopal', "verma's", 'own', 'adaptation', 'if', 'you', 'think', 'like', 'that', 'you', 'will', 'like', 'this', 'movie', 'br', 'br', 'well', 'this', 'movie', 'only', 'depends', 'on', 'the', 'viewer', 'and', 'on', 'his', 'judgement', 'whether', 'he', 'she', 'thinks', 'this', 'movie', 'is', 'total', 'copy',

In [9]:
MAX_SEQUENCE_LENGTH = max(
    [max(map(len, train_sequences)), max(map(len, test_sequences))]
)

In [10]:
MAX_SEQUENCE_LENGTH

2493

In [11]:
train_data = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)

In [25]:
len(train_data[0])

2493

In [26]:
print([tokenizer.index_word[k] for k in train_data[0]])

KeyError: np.int32(0)

In [28]:
from pprint import pprint

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [13]:
train_data

array([[   0,    0,    0, ...,    1,  168,   55],
       [   0,    0,    0, ...,   20,   11,   27],
       [   0,    0,    0, ...,    5, 7136, 1906],
       ...,
       [   0,    0,    0, ...,   40,  136,   54],
       [   0,    0,    0, ..., 1286,    1,  306],
       [   0,    0,    0, ...,   66,  167,    9]],
      shape=(75000, 2493), dtype=int32)

# Train a classifier with Word Embeddings

In [14]:
countries_wiki = KeyedVectors.load("wiki-countries.w2v")

In [15]:
embedding_layer = utils.make_embedding_layer(
    countries_wiki, tokenizer, MAX_SEQUENCE_LENGTH
)
countries_wiki_model = Sequential(
    [
        Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
        embedding_layer,
        GlobalAveragePooling1D(),
        Dense(128, activation="relu"),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)
countries_wiki_model.compile(
    loss="binary_crossentropy", optimizer=Adam(), metrics=["accuracy"]
)

/Users/ankster/.virtualenvs/ML_AI/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
countries_wiki_history = countries_wiki_model.fit(
    train_data,
    dfTrain["label"].values,
    validation_data=(test_data, xts["label"].values),
    batch_size=64,
    epochs=30,
)

Epoch 1/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -12956.7461 - val_accuracy: 0.5000 - val_loss: 51335.6953
Epoch 2/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -172890.3906 - val_accuracy: 0.5000 - val_loss: 344508.3438
Epoch 3/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -632292.8750 - val_accuracy: 0.5000 - val_loss: 974820.5625
Epoch 4/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -1460444.3750 - val_accuracy: 0.5000 - val_loss: 1999515.7500
Epoch 5/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -2702006.7500 - val_accuracy: 0.5000 - val_loss: 3453224.2500
Epoch 6/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -4401119.0000 - val_accuracy: 0.5000 - val_loss: 5390154.0000
Epoch 7/30
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1667 - loss: -6609110.5000 - val_accuracy: 0.5000 - val_loss: 7857599.5000
Epoch 8

In [ ]:
countries_wiki_history.history

In [17]:
dir(countries_wiki_history)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_api_export_path',
 '_api_export_symbol_id',
 '_model',
 'epoch',
 'history',
 'model',
 'on_batch_begin',
 'on_batch_end',
 'on_epoch_begin',
 'on_epoch_end',
 'on_predict_batch_begin',
 'on_predict_batch_end',
 'on_predict_begin',
 'on_predict_end',
 'on_test_batch_begin',
 'on_test_batch_end',
 'on_test_begin',
 'on_test_end',
 'on_train_batch_begin',
 'on_train_batch_end',
 'on_train_begin',
 'on_train_end',
 'params',
 'set_model',
 'set_params']

In [19]:
countries_wiki_history.history["accuracy"], countries_wiki_history.history[
    "val_accuracy"
]

([0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204,
  0.1666666716337204],
 [0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5,
  0.5])

In [21]:
pd.DataFrame(countries_wiki_history.history)

,accuracy,loss,val_accuracy,val_loss
0,0.166667,-1.295675e+04,0.5,5.133570e+04
1,0.166667,-1.728904e+05,0.5,3.445083e+05
2,0.166667,-6.322929e+05,0.5,9.748206e+05
3,0.166667,-1.460444e+06,0.5,1.999516e+06
4,0.166667,-2.702007e+06,0.5,3.453224e+06
5,0.166667,-4.401119e+06,0.5,5.390154e+06
6,0.166667,-6.609110e+06,0.5,7.857600e+06
7,0.166667,-9.378516e+06,0.5,1.091084e+07
8,0.166667,-1.276159e+07,0.5,1.459320e+07
9,0.166667,-1.679863e+07,0.5,1.895186e+07


# Train with a different set of word embeddings

## GloVe: Global Vectors for Word Representation
### Download [here](http://nlp.stanford.edu/data/glove.6B.zip)

In [ ]:
glove_wiki = KeyedVectors.load_word2vec_format(
    "data/glove.6B/glove.6B.300d.txt", binary=False, no_header=True
)

In [ ]:
embedding_layer = utils.make_embedding_layer(glove_wiki, tokenizer, MAX_SEQUENCE_LENGTH)

glove_model = Sequential(
    [
        Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
        embedding_layer,
        GlobalAveragePooling1D(),
        Dense(128, activation="relu"),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)
glove_model.compile(loss="binary_crossentropy", optimizer=Adam(), metrics=["accuracy"])

In [ ]:
glove_history = glove_model.fit(
    train_data,
    dfTrain["label"].values,
    validation_data=(test_data, xts["label"].values),
    batch_size=32,
    epochs=30,
)

In [ ]:
plt.plot(countries_wiki_history.history["val_accuracy"], label="Countries Wiki")
plt.plot(glove_history.history["val_accuracy"], label="All Wiki")
plt.legend()